# Gather gold argument mining corporaDownloads and preprocesses the 5 gold corpora (AbstRCT, arg-microtext, CDCP, PERSPECTRUM, AAEC) into `phase2_data/unified/`, the training input format used by Phase 2-beta.External downloads are noted in each cell (Zenodo, GitHub, Kaggle links).

In [ ]:
%%bash
mkdir -p ~/argument-aware-rag/phase2_data/raw \
         ~/argument-aware-rag/phase2_data/unified \
         ~/argument-aware-rag/phase2_data/silver
ls -la ~/argument-aware-rag/phase2_data/

In [ ]:
%%bash
BASE=~/argument-aware-rag/phase2_data
mkdir -p $BASE/raw $BASE/unified $BASE/silver
echo "Phase 2 data root: $BASE"
ls -la $BASE

In [ ]:
%%bash
BASE=~/argument-aware-rag/phase2_data/raw/liararg
mkdir -p $BASE
ln -sf ~/data/train.jsonl $BASE/train.jsonl
ln -sf ~/data/val.jsonl   $BASE/val.jsonl
ln -sf ~/data/test.jsonl  $BASE/test.jsonl
ls -lh $BASE
wc -l $BASE/*.jsonl

In [ ]:
%%bash
cd ~/argument-aware-rag/phase2_data/raw
rm -rf abstrct
git clone https://gitlab.com/tomaye/abstrct.git
echo ""
echo "=== AbstRCT contents ==="
ls abstrct/
find abstrct -maxdepth 3 -type d | head -10

In [ ]:
%%bash
cd ~/argument-aware-rag/phase2_data/raw
rm -rf microtext
git clone https://github.com/peldszus/arg-microtexts.git microtext
echo ""
ls microtext/
ls microtext/corpus | head -20
echo "Total English XML files: $(ls microtext/corpus/en | wc -l)"

In [ ]:
%%bash
cd ~/argument-aware-rag/phase2_data/raw
rm -rf perspectrum
git clone https://github.com/CogComp/perspectrum.git
echo ""
ls perspectrum/data/ 2>/dev/null || ls perspectrum/

In [ ]:
%%bash
cd ~/argument-aware-rag/phase2_data/raw
mkdir -p cdcp && cd cdcp
# Joonsuk Park's CDCP redistribution; URL has moved several times — adjust if 404
wget -q "https://facultystaff.richmond.edu/~jpark/data/cdcp_acl17.zip" -O cdcp.zip \
  || echo "Direct URL failed — fall back to HF datasets in next cell"
[ -f cdcp.zip ] && unzip -o cdcp.zip && ls

In [ ]:
%%bash
cd ~/argument-aware-rag/phase2_data/raw/echr
python3 -m gdown "1B06VVBpo1FF0aNTSkNDWFTaAgT_osbZJ" -O echr_download
echo ""
ls -lh echr_download

In [ ]:
%%bash
cd ~/argument-aware-rag/phase2_data/raw/echr
mkdir -p extracted
# Try common archive formats
unzip -q echr_download -d extracted 2>/dev/null && echo "→ unzipped" \
  || tar xzf echr_download -C extracted 2>/dev/null && echo "→ untarred (gz)" \
  || tar xjf echr_download -C extracted 2>/dev/null && echo "→ untarred (bz2)" \
  || tar xf echr_download -C extracted 2>/dev/null && echo "→ untarred (plain)" \
  || echo "Unknown archive — try opening manually"
echo ""
echo "=== contents ==="
ls extracted/ 2>/dev/null
echo ""
echo "=== file tree (first 20 entries) ==="
find extracted -maxdepth 3 2>/dev/null | head -20

In [ ]:
%%bash
     cd ~/argument-aware-rag/phase2_data/raw
     unzip brat-project-final.zip -d aaec


In [ ]:
%%bash
BASE=~/argument-aware-rag/phase2_data/raw
echo "============================================================"
echo "=== LIARArg (one row of train.jsonl) ==="
echo "============================================================"
head -1 $BASE/liararg/train.jsonl | python3 -m json.tool 2>/dev/null | head -30 || head -c 500 $BASE/liararg/train.jsonl

echo ""
echo "============================================================"
echo "=== AbstRCT (file tree + one annotation file) ==="
echo "============================================================"
find $BASE/abstrct -maxdepth 3 -type d | head -10
echo "--- first .ann file ---"
find $BASE/abstrct -name "*.ann" | head -1 | xargs -I{} bash -c 'echo "FILE: {}"; head -20 "{}"'
echo "--- matching .txt file ---"
find $BASE/abstrct -name "*.txt" | head -1 | xargs -I{} bash -c 'echo "FILE: {}"; head -3 "{}"'

echo ""
echo "============================================================"
echo "=== Microtext (one XML file) ==="
echo "============================================================"
find $BASE/microtext/corpus -name "*.xml" 2>/dev/null | head -1 | xargs -I{} bash -c 'echo "FILE: {}"; head -50 "{}"'

echo ""
echo "============================================================"
echo "=== PERSPECTRUM (file listing + key JSON sample) ==="
echo "============================================================"
ls $BASE/perspectrum/data/ 2>/dev/null || ls $BASE/perspectrum/
for f in $BASE/perspectrum/data/*.json $BASE/perspectrum/*.json; do
    [ -f "$f" ] && echo "FILE: $f" && head -c 800 "$f" && echo "..." && break
done

echo ""
echo "============================================================"
echo "=== CDCP (whatever made it through) ==="
echo "============================================================"
ls $BASE/cdcp/ 2>/dev/null

echo ""
echo "============================================================"
echo "=== ECHR (one JSON file) ==="
echo "============================================================"
ls $BASE/echr/extracted/all-data/ | head -3
FIRST=$(ls $BASE/echr/extracted/all-data/*.json | head -1)
echo "FILE: $FIRST"
python3 -m json.tool "$FIRST" 2>/dev/null | head -40 || head -c 1000 "$FIRST"

In [ ]:
import os, re, json, glob
from pathlib import Path

BASE = Path.home() / "argument-aware-rag/phase2_data"
RAW = BASE / "raw/abstrct/AbstRCT_corpus/data"
OUT = BASE / "unified"
OUT.mkdir(parents=True, exist_ok=True)

# Map AbstRCT brat labels to our schema
COMPONENT_MAP = {
    "Premise":    "premise",
    "Claim":      "claim",
    "MajorClaim": "claim",   # collapse MajorClaim into claim
}
RELATION_MAP = {
    "Support":        "support",
    "Attack":         "attack",
    "Partial-Attack": "pattack",
}

INSTRUCTION = ("Extract all argument components and relations from the text. "
               "Output strict JSON with claim_components, premise_components, "
               "citation_components, and relations.")

def parse_ann(ann_path: Path) -> dict:
    """Return dict with 'components' (id→{type,text,start,end}) and 'relations' list."""
    components = {}  # T1 → {...}
    relations = []
    with open(ann_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            tid = parts[0]
            if tid.startswith("T"):
                # T1\tPremise 1202 1335\tIntent-to-treat analysis...
                meta = parts[1].split()
                ctype = meta[0]
                start, end = int(meta[1]), int(meta[-1])
                text = parts[2] if len(parts) > 2 else ""
                if ctype in COMPONENT_MAP:
                    components[tid] = {
                        "type": COMPONENT_MAP[ctype],
                        "text": text, "start": start, "end": end,
                    }
            elif tid.startswith("R"):
                # R1\tSupport Arg1:T1 Arg2:T4\t
                meta = parts[1].split()
                rtype = meta[0]
                src = meta[1].split(":")[1]
                tgt = meta[2].split(":")[1]
                if rtype in RELATION_MAP:
                    relations.append({
                        "src_tid": src, "tgt_tid": tgt,
                        "type": RELATION_MAP[rtype],
                    })
    return {"components": components, "relations": relations}

def build_record(txt_path: Path, ann_path: Path, split: str, domain: str):
    text = txt_path.read_text(encoding="utf-8", errors="replace").strip()
    parsed = parse_ann(ann_path)

    # Assign stable integer IDs (T1 → 1, T2 → 2, ...) — required by ArgStructureDict
    tid_to_int = {tid: i + 1 for i, tid in enumerate(sorted(parsed["components"]))}

    claims = []
    premises = []
    for tid, c in parsed["components"].items():
        cid = tid_to_int[tid]
        component_dict = {"id": cid, "type": c["type"], "text": c["text"]}
        if c["type"] == "claim":
            claims.append(component_dict)
        elif c["type"] == "premise":
            premises.append(component_dict)

    relations = []
    for r in parsed["relations"]:
        if r["src_tid"] not in tid_to_int or r["tgt_tid"] not in tid_to_int:
            continue
        relations.append({
            "src": tid_to_int[r["src_tid"]],
            "tgt": tid_to_int[r["tgt_tid"]],
            "type": r["type"],
        })

    return {
        "instruction": INSTRUCTION,
        "input": text,
        "reasoning": "",
        "output": {
            "claim_components": claims,
            "premise_components": premises,
            "citation_components": [],
            "relations": relations,
        },
        "source_dataset": "abstrct",
        "label_kind": "gold",
        "split": split,
        "domain": f"abstrct_{domain}",
    }

# Walk train/dev/test directories (dev → val in our schema)
SPLIT_MAP = {"train": "train", "dev": "val", "test": "test"}
records_by_split = {"train": [], "val": [], "test": []}

for split_dir, our_split in SPLIT_MAP.items():
    split_path = RAW / split_dir
    if not split_path.exists():
        print(f"  [warn] missing {split_path}")
        continue
    for disease_dir in sorted(split_path.iterdir()):
        if not disease_dir.is_dir():
            continue
        disease = disease_dir.name.replace(f"_{split_dir}", "")
        for txt_file in sorted(disease_dir.glob("*.txt")):
            ann_file = txt_file.with_suffix(".ann")
            if not ann_file.exists():
                continue
            try:
                rec = build_record(txt_file, ann_file, our_split, disease)
                records_by_split[our_split].append(rec)
            except Exception as e:
                print(f"  [skip] {txt_file.name}: {e}")

# Write per-split JSONLs
for split, recs in records_by_split.items():
    out_path = OUT / f"abstrct_{split}.jsonl"
    with open(out_path, "w") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  → {out_path.name}: {len(recs)} records")

# Quick sanity-check the schema on one record
print("\n=== sample record (first train) ===")
print(json.dumps(records_by_split["train"][0], indent=2, ensure_ascii=False)[:1500])

In [ ]:
import json, re, hashlib
import xml.etree.ElementTree as ET
from pathlib import Path

BASE = Path.home() / "argument-aware-rag/phase2_data"
RAW = BASE / "raw/microtext/corpus/en"
OUT = BASE / "unified"
OUT.mkdir(parents=True, exist_ok=True)

EDGE_MAP = {
    "sup": "support",
    "reb": "attack",
    "exa": "psupport",   # 'example' — partial support
    "und": "pattack",    # 'undercut' — partial attack
}

INSTRUCTION = ("Extract all argument components and relations from the text. "
               "Output strict JSON with claim_components, premise_components, "
               "citation_components, and relations.")

def parse_microtext_xml(xml_path: Path) -> dict:
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # EDUs: text spans
    edu_text = {}
    for edu in root.findall("edu"):
        edu_text[edu.attrib["id"]] = (edu.text or "").strip()

    # ADUs: argument units (typed pro/opp)
    adu_type = {}
    for adu in root.findall("adu"):
        adu_type[adu.attrib["id"]] = adu.attrib.get("type", "")

    # Edges: build seg (EDU→ADU) and content (ADU↔ADU) edges
    adu_text = {}              # adu_id → joined EDU text(s)
    arg_edges = []             # (src_adu, tgt_adu, type)
    for edge in root.findall("edge"):
        src, tgt, etype = edge.attrib["src"], edge.attrib["trg"], edge.attrib["type"]
        if etype == "seg":     # EDU → ADU: assign text to the ADU
            if src in edu_text and tgt in adu_type:
                adu_text[tgt] = (adu_text.get(tgt, "") + " " + edu_text[src]).strip()
        elif etype in EDGE_MAP:
            arg_edges.append((src, tgt, EDGE_MAP[etype]))

    # Identify the central claim: the ADU referenced by the most incoming
    # argument edges (everything supports/attacks it), or the one with no
    # outgoing arg edges if tied.
    in_count = {adu: 0 for adu in adu_type}
    out_count = {adu: 0 for adu in adu_type}
    for s, t, _ in arg_edges:
        if t in in_count: in_count[t] += 1
        if s in out_count: out_count[s] += 1
    # Central claim = highest in_count, break tie with lowest out_count
    candidates = sorted(adu_type, key=lambda a: (-in_count[a], out_count[a]))
    claim_adu = candidates[0] if candidates else None

    # Build text + structure
    full_text = " ".join(edu_text[eid] for eid in sorted(edu_text.keys()))

    # Stable integer IDs (a1→1, a2→2 ...)
    adu_order = sorted(adu_type)
    adu_to_int = {adu: i + 1 for i, adu in enumerate(adu_order)}

    claims, premises = [], []
    for adu in adu_order:
        if adu not in adu_text:
            continue
        comp = {
            "id": adu_to_int[adu],
            "type": "claim" if adu == claim_adu else "premise",
            "text": adu_text[adu],
        }
        if adu == claim_adu:
            claims.append(comp)
        else:
            premises.append(comp)

    relations = []
    for s, t, rtype in arg_edges:
        if s not in adu_to_int or t not in adu_to_int:
            continue
        relations.append({"src": adu_to_int[s], "tgt": adu_to_int[t], "type": rtype})

    topic = root.attrib.get("topic_id", "")
    return {
        "text": full_text,
        "claims": claims,
        "premises": premises,
        "relations": relations,
        "topic": topic,
    }

def split_for(filename: str) -> str:
    """Deterministic 80/10/10 split based on filename hash."""
    h = int(hashlib.md5(filename.encode()).hexdigest(), 16) % 10
    if h < 8: return "train"
    if h < 9: return "val"
    return "test"

records_by_split = {"train": [], "val": [], "test": []}
xml_files = sorted(RAW.glob("*.xml"))
print(f"Found {len(xml_files)} EN XML files")

for xml_path in xml_files:
    try:
        parsed = parse_microtext_xml(xml_path)
        split = split_for(xml_path.name)
        rec = {
            "instruction": INSTRUCTION,
            "input": parsed["text"],
            "reasoning": "",
            "output": {
                "claim_components": parsed["claims"],
                "premise_components": parsed["premises"],
                "citation_components": [],
                "relations": parsed["relations"],
            },
            "source_dataset": "microtext",
            "label_kind": "gold",
            "split": split,
            "domain": f"microtext_{parsed['topic'] or 'general'}",
        }
        records_by_split[split].append(rec)
    except Exception as e:
        print(f"  [skip] {xml_path.name}: {e}")

for split, recs in records_by_split.items():
    out_path = OUT / f"microtext_{split}.jsonl"
    with open(out_path, "w") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  → {out_path.name}: {len(recs)} records")

print("\n=== sample record (first train) ===")
print(json.dumps(records_by_split["train"][0], indent=2, ensure_ascii=False)[:1500])

In [ ]:
%%bash
BASE=~/argument-aware-rag/phase2_data/raw

echo "============================================================"
echo "=== PERSPECTRUM dataset/ structure ==="
echo "============================================================"
ls -la $BASE/perspectrum/data/dataset/ 2>/dev/null || ls -la $BASE/perspectrum/dataset/ 2>/dev/null
echo ""
echo "--- searching for the main claim/perspective/evidence files ---"
find $BASE/perspectrum -maxdepth 4 -name "claim*.json" -o -name "perspective*.json" -o -name "evidence*.json" 2>/dev/null | head -10
echo ""
echo "--- first claim file content ---"
CLAIM_FILE=$(find $BASE/perspectrum -maxdepth 4 -name "claim*.json" 2>/dev/null | head -1)
if [ -n "$CLAIM_FILE" ]; then
    echo "FILE: $CLAIM_FILE"
    head -c 1500 "$CLAIM_FILE"
    echo "..."
fi

echo ""
echo "============================================================"
echo "=== CDCP actual contents ==="
echo "============================================================"
find $BASE/cdcp -maxdepth 4 -type f 2>/dev/null | head -20
echo ""
echo "--- first .ann file if any ---"
find $BASE/cdcp -name "*.ann" 2>/dev/null | head -1 | xargs -I{} bash -c 'echo "FILE: {}"; head -20 "{}"'
echo ""
echo "--- first .txt if any ---"
find $BASE/cdcp -name "*.txt" 2>/dev/null | head -1 | xargs -I{} bash -c 'echo "FILE: {}"; head -5 "{}"'
echo ""
echo "--- readme if present ---"
find $BASE/cdcp -iname "readme*" 2>/dev/null | head -1 | xargs -I{} bash -c 'echo "FILE: {}"; head -30 "{}"'

In [ ]:
%%bash
BASE=~/argument-aware-rag/phase2_data/raw

echo "============================================================"
echo "=== PERSPECTRUM main file (perspectrum_with_answers) ==="
echo "============================================================"
python3 -c "
import json
with open('$BASE/perspectrum/data/dataset/perspectrum_with_answers_v1.0.json') as f:
    data = json.load(f)
print(f'Type: {type(data).__name__}')
print(f'Length: {len(data)}')
print(f'First entry keys: {list(data[0].keys()) if isinstance(data, list) else list(data.keys())[:10]}')
print(f'First entry (truncated):')
print(json.dumps(data[0] if isinstance(data, list) else list(data.values())[0], indent=2)[:1500])
print()
print('--- evidence pool sample ---')
with open('$BASE/perspectrum/data/dataset/evidence_pool_v1.0.json') as f:
    ev = json.load(f)
print(f'Evidence count: {len(ev)}')
print(json.dumps(ev[0] if isinstance(ev, list) else list(ev.values())[0], indent=2)[:500])
print()
print('--- perspective pool sample ---')
with open('$BASE/perspectrum/data/dataset/perspective_pool_v1.0.json') as f:
    pp = json.load(f)
print(f'Perspective count: {len(pp)}')
print(json.dumps(pp[0] if isinstance(pp, list) else list(pp.values())[0], indent=2)[:500])
print()
print('--- split file ---')
with open('$BASE/perspectrum/data/dataset/dataset_split_v1.0.json') as f:
    sp = json.load(f)
print(f'split keys: {list(sp.keys()) if isinstance(sp, dict) else \"list\"}')
print(json.dumps(sp, indent=2)[:300])
"

echo ""
echo "============================================================"
echo "=== CDCP .ann.json schema ==="
echo "============================================================"
ANN=$(find $BASE/cdcp -name "*.ann.json" | head -1)
TXT=${ANN%.ann.json}.txt
echo "TXT: $TXT"
head -c 500 "$TXT"
echo ""
echo ""
echo "ANN: $ANN"
python3 -c "
import json
with open('$ANN') as f:
    d = json.load(f)
print(f'Keys: {list(d.keys())}')
print(json.dumps(d, indent=2)[:2000])
"

In [ ]:
import json
from pathlib import Path

BASE = Path.home() / "argument-aware-rag/phase2_data"
RAW = BASE / "raw/perspectrum/data/dataset"
OUT = BASE / "unified"

INSTRUCTION = ("Extract all argument components and relations from the text. "
               "Output strict JSON with claim_components, premise_components, "
               "citation_components, and relations.")

STANCE_MAP = {  # 3-label stance → our relation type
    "SUPPORT":   "support",
    "UNDERMINE": "attack",
}

# Load the three reference pools
with open(RAW / "perspective_pool_v1.0.json") as f:
    persp_pool = {int(p["pId"]): p["text"] for p in json.load(f)}
with open(RAW / "evidence_pool_v1.0.json") as f:
    ev_pool = {int(e["eId"]): e["text"] for e in json.load(f)}
with open(RAW / "dataset_split_v1.0.json") as f:
    split_map = json.load(f)   # cId (str) → "train"|"dev"|"test"
with open(RAW / "perspectrum_with_answers_v1.0.json") as f:
    claims = json.load(f)

records_by_split = {"train": [], "val": [], "test": []}

for claim_data in claims:
    cid = int(claim_data["cId"])
    split = split_map.get(str(cid), "train")
    if split == "dev":
        split = "val"

    main_claim_text = claim_data["text"]
    components = [{"id": 1, "type": "claim", "text": main_claim_text}]
    premises, citations, relations = [], [], []
    next_id = 2

    for persp in claim_data.get("perspectives", []):
        rel_type = STANCE_MAP.get(persp.get("stance_label_3", ""))
        if not rel_type:
            continue
        pids = persp.get("pids", [])
        if not pids:
            continue
        ptext = persp_pool.get(pids[0], "")
        if not ptext:
            continue
        p_int = next_id; next_id += 1
        premises.append({"id": p_int, "type": "premise", "text": ptext})
        relations.append({"src": p_int, "tgt": 1, "type": rel_type})

        # Evidence sentences support their perspective
        for eid in persp.get("evidence", []):
            etext = ev_pool.get(eid, "")
            if not etext:
                continue
            e_int = next_id; next_id += 1
            citations.append({"id": e_int, "type": "citation", "text": etext})
            relations.append({"src": e_int, "tgt": p_int, "type": "support"})

    # Concatenate naturally (no separator markers — student should learn from prose)
    input_parts = [main_claim_text] + [p["text"] for p in premises] + [c["text"] for c in citations]
    input_text = ". ".join(s.rstrip(".") for s in input_parts) + "."

    records_by_split[split].append({
        "instruction": INSTRUCTION,
        "input": input_text,
        "reasoning": "",
        "output": {
            "claim_components": components,
            "premise_components": premises,
            "citation_components": citations,
            "relations": relations,
        },
        "source_dataset": "perspectrum",
        "label_kind": "gold",
        "split": split,
        "domain": "perspectrum",
    })

for split, recs in records_by_split.items():
    out_path = OUT / f"perspectrum_{split}.jsonl"
    with open(out_path, "w") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  → {out_path.name}: {len(recs)} records")

print("\n=== sample record (first train) ===")
print(json.dumps(records_by_split["train"][0], indent=2, ensure_ascii=False)[:1800])

In [ ]:
import json, hashlib
from pathlib import Path

BASE = Path.home() / "argument-aware-rag/phase2_data"
RAW = BASE / "raw/cdcp/cdcp"
OUT = BASE / "unified"

LABEL_MAP = {  # CDCP proposition label → our component kind
    "policy":    "claim",
    "value":     "claim",
    "fact":      "premise",
    "testimony": "premise",
    "reference": "citation",
}

INSTRUCTION = ("Extract all argument components and relations from the text. "
               "Output strict JSON with claim_components, premise_components, "
               "citation_components, and relations.")

def split_for(filename: str) -> str:
    """Deterministic 80/20 for splitting the train dir into train+val."""
    h = int(hashlib.md5(filename.encode()).hexdigest(), 16) % 10
    return "val" if h == 0 else "train"

records_by_split = {"train": [], "val": [], "test": []}

for split_dir_name in ("train", "test"):
    split_dir = RAW / split_dir_name
    if not split_dir.exists():
        print(f"  [warn] missing {split_dir}")
        continue

    for ann_path in sorted(split_dir.glob("*.ann.json")):
        base_name = ann_path.name.replace(".ann.json", "")
        txt_path = split_dir / f"{base_name}.txt"
        if not txt_path.exists():
            continue

        text = txt_path.read_text(encoding="utf-8", errors="replace")
        ann = json.loads(ann_path.read_text())

        prop_labels = ann.get("prop_labels", [])
        prop_offsets = ann.get("prop_offsets", [])

        claims, premises, citations = [], [], []
        prop_int_id = {}
        next_id = 1
        for idx, (label, offsets) in enumerate(zip(prop_labels, prop_offsets)):
            kind = LABEL_MAP.get(label)
            if not kind:
                continue
            start, end = offsets[0], offsets[1]
            comp = {"id": next_id, "type": kind, "text": text[start:end].strip()}
            prop_int_id[idx] = next_id
            (claims if kind == "claim" else premises if kind == "premise" else citations).append(comp)
            next_id += 1

        relations = []
        # reasons: [[src_start, src_end], tgt_idx] — supports
        for entry in (ann.get("reasons", []) + ann.get("evidences", [])):
            src_range, tgt_idx = entry
            if not (isinstance(src_range, list) and len(src_range) == 2):
                continue
            src_start, src_end = src_range
            for src_idx in range(src_start, src_end + 1):
                if src_idx in prop_int_id and tgt_idx in prop_int_id:
                    relations.append({
                        "src": prop_int_id[src_idx],
                        "tgt": prop_int_id[tgt_idx],
                        "type": "support",
                    })

        split = "test" if split_dir_name == "test" else split_for(base_name)
        records_by_split[split].append({
            "instruction": INSTRUCTION,
            "input": text,
            "reasoning": "",
            "output": {
                "claim_components": claims,
                "premise_components": premises,
                "citation_components": citations,
                "relations": relations,
            },
            "source_dataset": "cdcp",
            "label_kind": "gold",
            "split": split,
            "domain": "cdcp",
        })

for split, recs in records_by_split.items():
    out_path = OUT / f"cdcp_{split}.jsonl"
    with open(out_path, "w") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  → {out_path.name}: {len(recs)} records")

print("\n=== sample record (first train) ===")
print(json.dumps(records_by_split["train"][0], indent=2, ensure_ascii=False)[:1500])

In [ ]:
%%bash
OUT=~/argument-aware-rag/phase2_data/unified
echo "=== UNIFIED GOLD CORPORA ==="
for f in $OUT/*.jsonl; do
    n=$(wc -l < "$f")
    name=$(basename "$f")
    printf "  %-30s  %5d records\n" "$name" "$n"
done
echo ""
echo "Total gold records: $(cat $OUT/*.jsonl | wc -l)"

In [ ]:
import json
from pathlib import Path

BASE = Path.home() / "argument-aware-rag/phase2_data"
RAW = BASE / "raw/liararg"
OUT = BASE / "unified"

INSTRUCTION = ("Extract all argument components and relations from the text. "
               "Output strict JSON with claim_components, premise_components, "
               "citation_components, and relations.")

def liararg_row_to_record(row: dict, split: str) -> dict:
    def zip_components(ids, texts, kind):
        return [{"id": int(cid), "type": kind, "text": str(t)}
                for cid, t in zip(ids or [], texts or [])]

    claims    = zip_components(row.get("claim_ids"),    row.get("claim_texts"),    "claim")
    premises  = zip_components(row.get("premise_ids"),  row.get("premise_texts"),  "premise")
    citations = zip_components(row.get("citation_ids"), row.get("citation_texts"), "citation")

    relations = []
    for rtype in ("support", "attack", "psupport", "pattack"):
        for pair in (row.get(f"{rtype}_relations") or []):
            if len(pair) >= 2:
                relations.append({"src": int(pair[0]), "tgt": int(pair[1]), "type": rtype})

    input_text = row.get("full_text") or row.get("summary") or row.get("statement", "")

    return {
        "instruction": INSTRUCTION,
        "input": str(input_text),
        "reasoning": "",
        "output": {
            "claim_components": claims,
            "premise_components": premises,
            "citation_components": citations,
            "relations": relations,
        },
        "source_dataset": "liararg",
        "label_kind": "gold",
        "split": split,
        "domain": "politics",
        # Extras useful at integration time but ignored by the student
        "liararg_row_id": int(row.get("id", -1)),
        "liararg_label": row.get("label", ""),
    }

records_by_split = {"train": [], "val": [], "test": []}
for split_name in ("train", "val", "test"):
    src = RAW / f"{split_name}.jsonl"
    if not src.exists():
        print(f"  [warn] missing {src}")
        continue
    with open(src) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records_by_split[split_name].append(
                liararg_row_to_record(json.loads(line), split_name)
            )

for split, recs in records_by_split.items():
    out_path = OUT / f"liararg_{split}.jsonl"
    with open(out_path, "w") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  → {out_path.name}: {len(recs)} records")

print(f"\nTotal: {sum(len(r) for r in records_by_split.values())} records")
print("\n=== sample record (first test) ===")
print(json.dumps(records_by_split["test"][0], indent=2, ensure_ascii=False)[:1500])